# La Grulla Blanca — TIERRA HEAVY / S4 → S5 · V08 Strict Canonical Chain

Continúa **únicamente la raíz Tierra** desde el checkpoint S4 validado.

Autoridad de reanudación:
- `TECHNIQUES_CAMPAIGN_V01/TIERRA_S4_TECHNIQUES_HEAVY_CHECKPOINT.zip`
- SHA-256 obligatorio: `17b1215b41e91768e9a0e1aff2726d0e160fa15b49044714e89fe6ae669a5ef7`
- Estado conocido al cierre de S3→S4: técnicas completas `golpe_montana`, `piel_cobre` y `palma_ardiente` en el estado global de campaña.

Selección segura:
1. Busca primero el artefacto canónico S4 dentro del Output montado del notebook anterior.
2. Si no está montado, intenta el enlace directo del Output anterior (requiere Internet y puede expirar).
3. **No continúa** si el SHA-256 no coincide exactamente.
4. El motor decide desde `state.json` qué trabajo pendiente de Tierra debe continuar; no se fuerzan fases ni técnicas manualmente.

Si vuelve a ceder por presupuesto genera:
`TIERRA_S5_TECHNIQUES_HEAVY_CHECKPOINT.zip`

Si Tierra termina genera además:
`RESULTS_TIERRA_HEAVY.zip`

Sin promoción CANON automática; resultados para revisión humana.


In [ ]:
RUN_MODE = "NUMERIC_SEARCH"
RESUME = True
ENTRYPOINT = "tierra"
EXPECTED_RESUME_SHA256 = "17b1215b41e91768e9a0e1aff2726d0e160fa15b49044714e89fe6ae669a5ef7"
EXPECTED_RESUME_BASENAME = "TIERRA_S4_TECHNIQUES_HEAVY_CHECKPOINT.zip"
EXPECTED_RESUME_PARENT = "TECHNIQUES_CAMPAIGN_V01"
CHECKPOINT_URL_FALLBACK = 'https://www.kaggleusercontent.com/kf/355214082/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..pnGtVqdeLbJAWe3BHYOI6A.VQUW7TEIUChwPdfmZt7319AdOKmtgO72TRefc3ec07zeYDnd0aqRAY3n3K5nU0JNsHpf6sYZRwHZBUW9Jyy_VbgXUeNaj5GAGooqeUBNmMI-ZW-CLwfAgQeQyZPVyjqFurdv2WLynVeM3JphGIs3NH8NEH4hCphbFzQWx0hFHkfFUmCtFfjhV0NaCdh6JU9F2k8ek8mzs1K-wJxuqUuLNlnuTzC2XNYBdhO4YaGP-OZpadwTwq1x3Fzhv_jWFiKhE_0OzCYOvoW5cyzQRlDqq11vS18xw4tl_COK_cAbg8NKwQhuEwX3nMBkuAhb_UuDy-moLpbJJJY9Dgz7azAOkYqFQGkwx3kA4vjLM8fdO_7Q5s_q5CUR-ezs7ULS7mpUGLFg3CgphwBPUGopIoSAzxbVhvf8Q8tHF8cTo9jJ2VYv_zsNc-etnjmkTQK8hXCXKpjPPMhavXI1FSfmDs4s2OrUBTm6o6WZjf2gng0tA7WC1s8Vbxnb7JGuUEca0BFMZzR3oxVqpcU7TpCrRI6YkmsTC589o5OQQoemYAUfdG0gYx3U663NH1mPtiJm4Gef_rvPPEFLb67cTStelDH9HL7Ij8cIeqIdldOC26I2ScM.HusaerDAFqghSd4qfyPT5Q/TECHNIQUES_CAMPAIGN_V01/TIERRA_S4_TECHNIQUES_HEAVY_CHECKPOINT.zip'

print("[CONFIG READY] RUN_MODE=", RUN_MODE, "| RESUME=", RESUME, "| ENTRYPOINT=", ENTRYPOINT, flush=True)
print("[V08 STRICT CANONICAL CHAIN] expected checkpoint:", EXPECTED_RESUME_BASENAME, flush=True)
print("[V08 STRICT CANONICAL CHAIN] expected SHA256:", EXPECTED_RESUME_SHA256, flush=True)


In [ ]:
import sys, hashlib, zipfile, json, os, subprocess, shutil, base64
from pathlib import Path
sys.dont_write_bytecode = True

INPUT_ROOT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

OFFICIAL_REPO = "https://github.com/Shein25/La-Grulla-Blanca.git"
ASSET_COMMIT = "ac184bf496277eda78493262a1255f245d5c1d97"
ASSET_BASE = "experimentos/balance_nuevo/techniques_kaggle_heavy_v0_4/assets"
EXPECTED_ZIP_SHA = "dd4032c12a574d071411a505565f970473baaa84c17d4302435a9a206938d63b"
EXPECTED_MANIFEST_SHA = "85465dd566690674b94eab91bdd0aebf1053aef76d8373335c5c94aada08bdf6"
PARTS = [f"TECHNIQUES_KAGGLE_HEAVY_INPUTS_V01_1_HOTFIX.part{i:03d}.b64" for i in range(1, 6)]

def digest(raw):
    return hashlib.sha256(raw).hexdigest()

def run_git(*args, cwd=None):
    cmd = ["git", *args]
    print("[GIT]", " ".join(cmd), flush=True)
    try:
        p = subprocess.run(
            cmd,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=None,  # deja visible el progreso de clone/checkout en Kaggle
            check=False,
            timeout=900,
        )
    except subprocess.TimeoutExpired as e:
        raise RuntimeError("GIT_ASSET_BOOTSTRAP_TIMEOUT_900S") from e
    if p.returncode:
        raise RuntimeError("GIT_ASSET_BOOTSTRAP_FAILED:returncode=" + str(p.returncode))
    stdout = (p.stdout or "").strip()
    if stdout:
        print(stdout, flush=True)
    print("[GIT OK]", args[0] if args else "git", flush=True)
    return stdout
def authenticate(root):
    manifest = root / "MANIFEST_SHA256.txt"
    if digest(manifest.read_bytes()) != EXPECTED_MANIFEST_SHA:
        raise RuntimeError("INPUT_PACKAGE_AUTHORITY_MISMATCH")
    names = set()
    for line in manifest.read_text(encoding="utf-8").splitlines():
        expected, name = line.split("  ", 1)
        if name in names or "/" in name or "\\" in name or ":" in name:
            raise RuntimeError("INVALID_INPUT_MANIFEST")
        names.add(name)
        if digest((root / name).read_bytes()) != expected:
            raise RuntimeError("INPUT_HASH_MISMATCH:" + name)
    actual = {p.name for p in root.iterdir() if p.is_file()} - {"MANIFEST_SHA256.txt"}
    if actual != names:
        raise RuntimeError("INPUT_FILE_SET_MISMATCH")
    return root

print("[BOOT 1/6] Python:", sys.version.split()[0], flush=True)
print("[BOOT 2/6] Kaggle input root:", INPUT_ROOT, flush=True)
checkpoint_names = sorted(
    str(p) for p in INPUT_ROOT.rglob("*")
    if "TECHNIQUES_HEAVY_CHECKPOINT" in p.name.upper()
)
print("[BOOT 2/6] Name-matched checkpoint inputs found:", len(checkpoint_names), flush=True)
for p in checkpoint_names[:10]:
    print("  -", p, flush=True)
print("[BOOT 2/6] Mounted dataset roots:", flush=True)
for p in sorted(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []:
    print("  -", p, flush=True)
print("[BOOT 3/6] Fetching authenticated lab assets from GitHub...", flush=True)

ASSET_REPO = WORKING / "La-Grulla-Blanca-Kaggle-Assets"
if ASSET_REPO.exists():
    shutil.rmtree(ASSET_REPO)

run_git("clone", "--progress", "--filter=blob:none", "--no-checkout", OFFICIAL_REPO, str(ASSET_REPO))
run_git("checkout", "--detach", ASSET_COMMIT, cwd=ASSET_REPO)

print("[GIT VERIFY] Reading HEAD independently...", flush=True)
try:
    actual_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=ASSET_REPO,
        text=True,
        stderr=subprocess.STDOUT,
        timeout=60,
    ).strip()
except subprocess.TimeoutExpired as e:
    raise RuntimeError("GIT_REV_PARSE_TIMEOUT") from e
print("[GIT VERIFY] actual_commit =", repr(actual_commit), flush=True)
print("[GIT VERIFY] expected_commit =", repr(ASSET_COMMIT), flush=True)
if actual_commit != ASSET_COMMIT:
    raise RuntimeError(f"ASSET_COMMIT_MISMATCH:{actual_commit}!={ASSET_COMMIT}")
print("[GIT VERIFY OK] Exact asset commit matched.", flush=True)

asset_dir = ASSET_REPO / ASSET_BASE
encoded = "".join((asset_dir / p).read_text(encoding="ascii") for p in PARTS)
zip_bytes = base64.b64decode(encoded, validate=True)
if digest(zip_bytes) != EXPECTED_ZIP_SHA:
    raise RuntimeError("GIT_ASSET_ZIP_SHA_MISMATCH")

selected = WORKING / "TECHNIQUES_KAGGLE_HEAVY_INPUTS_V01_1_HOTFIX.zip"
selected.write_bytes(zip_bytes)

INPUTS = WORKING / "TECHNIQUES_LAB_INPUTS_V01"
if INPUTS.exists():
    shutil.rmtree(INPUTS)

with zipfile.ZipFile(selected) as z:
    if z.testzip() is not None:
        raise RuntimeError("CORRUPTED_INPUT_ZIP")
    if any("/" in n or "\\" in n or ":" in n for n in z.namelist()):
        raise RuntimeError("INVALID_INPUT_ZIP_PATH")
    if sum(i.file_size for i in z.infolist()) > 32 * 1024 * 1024:
        raise RuntimeError("INPUT_ZIP_SIZE_LIMIT")
    INPUTS.mkdir()
    for name in z.namelist():
        (INPUTS / name).write_bytes(z.read(name))

print("[BOOT 4/6] Authenticating reconstructed lab package...", flush=True)
authenticate(INPUTS)
print("[BOOT 4/6 OK] Git asset commit authenticated:", ASSET_COMMIT, flush=True)
print("[BOOT 4/6 OK] LAB input package authenticated:", INPUTS, flush=True)


In [ ]:
print("[BOOT 5/6] Loading and authenticating Tierra S4 checkpoint...", flush=True)
import shutil, zipfile, hashlib, json, urllib.request
from pathlib import Path

INPUT_NOTEBOOKS_ROOT = Path("/kaggle/input/notebooks")
RESUME_INPUT_ROOT = WORKING / "TECHNIQUES_RESUME_INPUT"
if RESUME_INPUT_ROOT.exists():
    shutil.rmtree(RESUME_INPUT_ROOT)
RESUME_INPUT_ROOT.mkdir(parents=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

REQUIRED_CHECKPOINT_MEMBERS = {
    "MANIFEST_SHA256.txt",
    "state.json",
    "authority.json",
    "campaign_configuration.json",
}

def inspect_checkpoint(path):
    try:
        with zipfile.ZipFile(path) as z:
            if z.testzip() is not None:
                return None
            names = set(z.namelist())
            if not REQUIRED_CHECKPOINT_MEMBERS.issubset(names):
                return None
            state = json.loads(z.read("state.json"))
        return {"path": Path(path), "sha256": sha256_file(path), "state": state}
    except (zipfile.BadZipFile, json.JSONDecodeError, OSError, KeyError, ValueError):
        return None

matches = []
if INPUT_NOTEBOOKS_ROOT.exists():
    matches = sorted(
        p.resolve()
        for p in INPUT_NOTEBOOKS_ROOT.rglob(EXPECTED_RESUME_BASENAME)
        if p.parent.name == EXPECTED_RESUME_PARENT
    )

print("[CHECKPOINT LOOKUP] canonical relative artifact:", f"{EXPECTED_RESUME_PARENT}/{EXPECTED_RESUME_BASENAME}", flush=True)
for p in matches:
    print("[CHECKPOINT LOOKUP] match:", p, "| sha=", sha256_file(p)[:16], flush=True)

valid = [p for p in matches if sha256_file(p) == EXPECTED_RESUME_SHA256]

if not valid and CHECKPOINT_URL_FALLBACK:
    downloaded = WORKING / EXPECTED_RESUME_BASENAME
    print("[CHECKPOINT DOWNLOAD] Canonical mounted S4 not found with expected hash; trying prior Output URL...", flush=True)
    try:
        with urllib.request.urlopen(CHECKPOINT_URL_FALLBACK, timeout=120) as src, open(downloaded, "wb") as dst:
            shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
        print("[CHECKPOINT DOWNLOAD] downloaded MB:", round(downloaded.stat().st_size / 1024 / 1024, 2), flush=True)
        print("[CHECKPOINT DOWNLOAD] SHA256:", sha256_file(downloaded), flush=True)
        if sha256_file(downloaded) == EXPECTED_RESUME_SHA256:
            valid = [downloaded]
    except Exception as e:
        print("[CHECKPOINT DOWNLOAD] fallback unavailable:", type(e).__name__, str(e)[:300], flush=True)

if not valid:
    observed = [(str(p), sha256_file(p)) for p in matches]
    raise RuntimeError("EXPECTED_TIERRA_S4_CHECKPOINT_NOT_FOUND_OR_SHA_MISMATCH:" + repr(observed))

SOURCE_CHECKPOINT = sorted(valid, key=str)[0]
staged = RESUME_INPUT_ROOT / "TECHNIQUES_HEAVY_CHECKPOINT.zip"
shutil.copy2(SOURCE_CHECKPOINT, staged)

if sha256_file(staged) != EXPECTED_RESUME_SHA256:
    raise RuntimeError("STAGED_TIERRA_S4_SHA_MISMATCH")

info = inspect_checkpoint(staged)
if info is None:
    raise RuntimeError("STAGED_TIERRA_S4_STRUCTURE_INVALID")
state = info["state"]
root_status = state.get("roots", {}).get("tierra")
if root_status not in ("RUNNING", "COMPLETE"):
    raise RuntimeError("STAGED_CHECKPOINT_WRONG_TIERRA_STATUS:" + str(root_status))

phases = state.get("phases", {})
complete_techniques = sorted(
    key.split(":", 1)[1]
    for key, val in phases.items()
    if key.startswith("TECHNIQUE:") and isinstance(val, dict) and val.get("status") == "COMPLETE"
)
expected_completed = {"golpe_montana", "piel_cobre", "palma_ardiente"}
if not expected_completed.issubset(set(complete_techniques)):
    raise RuntimeError("TIERRA_S4_EXPECTED_COMPLETED_TECHNIQUES_MISSING:" + repr(complete_techniques))

print("[CHECKPOINT OK] Canonical/source:", SOURCE_CHECKPOINT, flush=True)
print("[CHECKPOINT OK] staged as:", staged, flush=True)
print("[CHECKPOINT OK] checkpoint_sequence:", state.get("checkpoint_sequence"), flush=True)
print("[CHECKPOINT OK] total_fights:", state.get("total_fights"), flush=True)
print("[CHECKPOINT OK] tierra status:", root_status, flush=True)
print("[CHECKPOINT OK] completed techniques:", complete_techniques, flush=True)
print("[CHECKPOINT OK] Size MB:", round(staged.stat().st_size / 1024 / 1024, 2), flush=True)
print("[CHECKPOINT OK] SHA256:", sha256_file(staged), flush=True)


In [ ]:
print("[BOOT 6/6] Preparing pinned CPU dependencies...", flush=True)
import subprocess
if sys.version_info < (3, 11):
    raise RuntimeError("PYTHON_3_11_OR_NEWER_REQUIRED")

DEPENDENCIES = json.loads(
    (INPUTS / "CAMPAIGN_CONFIG.json").read_text(encoding="utf-8")
)["DEPENDENCIES"]

DEPENDENCY_TARGET = WORKING / "TECHNIQUES_DEPENDENCIES_V01_1"
lock_file = DEPENDENCY_TARGET / "PINNED_DEPENDENCIES.json"

if not lock_file.exists() or json.loads(lock_file.read_text()) != DEPENDENCIES:
    if DEPENDENCY_TARGET.exists():
        shutil.rmtree(DEPENDENCY_TARGET)
    DEPENDENCY_TARGET.mkdir(parents=True, exist_ok=True)
    print("[PIP] Installing isolated pinned dependencies; this can take a few minutes...", flush=True)
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--disable-pip-version-check",
            "--quiet",
            "--ignore-installed",
            "--no-warn-script-location",
            "--target", str(DEPENDENCY_TARGET),
        ] + [k + "==" + v for k, v in DEPENDENCIES.items()],
        check=True,
    )
    lock_file.write_text(json.dumps(DEPENDENCIES), encoding="utf-8")

print("[BOOT COMPLETE] Pinned CPU dependencies ready. Simulation can start.", flush=True)


In [ ]:
print(f"[RUNNER START] scope={ENTRYPOINT} mode={RUN_MODE} resume={RESUME}", flush=True)
request = {
    "inputs": str(INPUTS),
    "working": str(WORKING),
    "input_root": str(RESUME_INPUT_ROOT),
    "RUN_MODE": RUN_MODE,
    "resume": RESUME,
    "scope": ENTRYPOINT,
}
request_file = WORKING / "TECHNIQUES_RUN_REQUEST_TIERRA_S4_TO_S5.json"
request_file.write_text(json.dumps(request), encoding="utf-8")

environment = dict(os.environ)
environment["PYTHONPATH"] = str(DEPENDENCY_TARGET) + os.pathsep + str(INPUTS)
environment["PYTHONDONTWRITEBYTECODE"] = "1"

command = [sys.executable, "-u", "-B", str(INPUTS / "campaign_entrypoint.py"), "--run", str(request_file)]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=environment) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    status = process.wait()

if status:
    raise RuntimeError("CAMPAIGN_FAILED: inspect the exact evidence above; TIERRA S4 checkpoint remains untouched")
print("Tierra S4 continuation finished/yielded cleanly.")


In [ ]:
# Compact the checkpoint conservatively:
# only raw finalist batches belonging to techniques already marked COMPLETE are removed.
import re, zipfile, hashlib, json

ARTIFACTS = WORKING / "TECHNIQUES_CAMPAIGN_V01"
SOURCE_CP = ARTIFACTS / "TECHNIQUES_HEAVY_CHECKPOINT.zip"
COMPACT_CP = ARTIFACTS / "TIERRA_S5_TECHNIQUES_HEAVY_CHECKPOINT.zip"

def _sha(raw):
    return hashlib.sha256(raw).hexdigest()

def compact_checkpoint(src, dst):
    with zipfile.ZipFile(src, "r") as zin:
        state = json.loads(zin.read("state.json"))
        phases = state.get("phases", {})
        completed = {
            key.split(":", 1)[1]
            for key, val in phases.items()
            if key.startswith("TECHNIQUE:")
            and isinstance(val, dict)
            and val.get("status") == "COMPLETE"
        }

        kept = []
        removed = []
        for info in zin.infolist():
            name = info.filename
            if name in {"MANIFEST_SHA256.txt", "COMPACTION_REPORT.json"}:
                continue
            parts = name.split("/")
            prune = (
                len(parts) >= 7
                and parts[0] == "roots"
                and parts[2] in completed
                and parts[3] == "finalists"
                and re.fullmatch(r"R\d+", parts[5] or "") is not None
            )
            if prune:
                removed.append((name, info.compress_size, info.file_size))
            else:
                kept.append((name, zin.read(name)))

    report = {
        "status": "SAFE_COMPLETED_TECHNIQUE_RAW_COMPACTION",
        "completed_techniques_detected": sorted(completed),
        "removed_files": len(removed),
        "removed_uncompressed_bytes": sum(x[2] for x in removed),
        "removed_original_zip_compressed_bytes": sum(x[1] for x in removed),
        "policy": (
            "Only raw finalist R<replicates>/ batches of TECHNIQUE:<id>=COMPLETE are removed. "
            "Summary JSON, human shortlist, studies, baselines, authority and state are retained."
        ),
    }
    kept.append((
        "COMPACTION_REPORT.json",
        json.dumps(report, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8"),
    ))

    manifest = [f"{_sha(data)}  {name}" for name, data in sorted(kept)]
    pending = dst.with_suffix(dst.suffix + ".pending")

    with zipfile.ZipFile(pending, "w", zipfile.ZIP_DEFLATED, compresslevel=3) as zout:
        for name, data in sorted(kept):
            zout.writestr(name, data)
        zout.writestr("MANIFEST_SHA256.txt", "\n".join(manifest) + "\n")

    pending.replace(dst)

    # Full integrity verification.
    with zipfile.ZipFile(dst) as z:
        names = z.namelist()
        if len(names) != len(set(names)) or z.testzip() is not None:
            raise RuntimeError("COMPACT_CHECKPOINT_CORRUPTED")
        listed = {}
        for line in z.read("MANIFEST_SHA256.txt").decode("utf-8").splitlines():
            h, name = line.split("  ", 1)
            if name in listed or name not in names or _sha(z.read(name)) != h:
                raise RuntimeError("COMPACT_CHECKPOINT_MANIFEST_MISMATCH:" + name)
            listed[name] = h
        if set(listed) != set(names) - {"MANIFEST_SHA256.txt"}:
            raise RuntimeError("COMPACT_CHECKPOINT_FILESET_MISMATCH")

    return report

if not SOURCE_CP.is_file():
    raise RuntimeError("CAMPAIGN_DID_NOT_PRODUCE_CHECKPOINT")

report = compact_checkpoint(SOURCE_CP, COMPACT_CP)

print("Checkpoint compacted safely.")
print("Completed techniques compacted:", report["completed_techniques_detected"])
print("Removed raw finalist files:", report["removed_files"])
print("Original MB:", round(SOURCE_CP.stat().st_size / 1024 / 1024, 2))
print("Compact MB:", round(COMPACT_CP.stat().st_size / 1024 / 1024, 2))
print("Compact SHA256:", hashlib.sha256(COMPACT_CP.read_bytes()).hexdigest())

from IPython.display import display, FileLink

print("\nDOWNLOAD FOR NEXT SESSION (TIERRA S5):")
display(FileLink(os.path.relpath(COMPACT_CP, Path.cwd())))

root_result = ARTIFACTS / "RESULTS_TIERRA_HEAVY.zip"
if root_result.is_file():
    print("\nTIERRA COMPLETE — DOWNLOAD RESULT (the S4 checkpoint remains only as archival/resume evidence):")
    display(FileLink(os.path.relpath(root_result, Path.cwd())))
else:
    print("\nTierra is not complete yet. Use TIERRA_S5_TECHNIQUES_HEAVY_CHECKPOINT.zip as the only checkpoint input in the next Tierra session.")


In [ ]:
from pathlib import Path

print("\n=== TIERRA S4 → S5 OUTPUT SUMMARY ===")
for p in [
    Path("/kaggle/working/TECHNIQUES_CAMPAIGN_V01/TIERRA_S5_TECHNIQUES_HEAVY_CHECKPOINT.zip"),
    Path("/kaggle/working/TECHNIQUES_CAMPAIGN_V01/RESULTS_TIERRA_HEAVY.zip"),
]:
    if p.is_file():
        print(f"READY: {p}  ({p.stat().st_size/1024/1024:.2f} MB)")
    else:
        print(f"NOT PRESENT: {p}")
